# BigEarthNet Cross-Modal Retrieval

Single Kaggle notebook for S1/S2 training, same- and cross-modal evaluation, experiment comparison, and Hugging Face Space export.

## 1. Packages and configuration

In [ ]:
import importlib.util, subprocess, sys
missing=[p for m,p in {"timm":"timm>=1.0.15","rasterio":"rasterio>=1.3","gradio":"gradio>=4.44","huggingface_hub":"huggingface_hub>=0.27","safetensors":"safetensors>=0.4"}.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])

import ast, hashlib, json, os, random, shutil, time, warnings, zipfile
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import rasterio
from rasterio.enums import Resampling
import timm, torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file as load_safetensors
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

@dataclass
class Config:
    DATA_ROOT: Path=Path("/kaggle/input/datasets/glitchr/bigearthnet-48k-subset")
    OUTPUT_DIR: Path=Path("/kaggle/working/outputs")
    MODEL_NAME: str="vit_base_patch8_224"
    EMBED_DIM: int=256
    IMAGE_SIZE: int=120
    LOCAL_PRETRAINED_PATH: str|None=None
    USE_HF_REBEN: bool=True
    HF_S1_REPO: str="BIFOLD-BigEarthNetv2-0/vit_base_patch8_224-s1-v0.1.1"
    HF_S2_REPO: str="BIFOLD-BigEarthNetv2-0/vit_base_patch8_224-s2-v0.1.1"
    HF_LOCAL_DIR: str|None=None  # Optional Kaggle input folder containing s1/ and s2/ model files.
    HF_TOKEN: str|None=None
    USE_GENERIC_PRETRAINED: bool=True
    TASK_MODE: str="all" # all/s1_to_s1/s2_to_s2/s1_to_s2/s2_to_s1
    BATCH_SIZE: int=64
    EVAL_BATCH_SIZE: int=128
    NUM_WORKERS: int=2
    EPOCHS: int=20
    LR: float=3e-4
    WEIGHT_DECAY: float=1e-4
    TEMPERATURE: float=.07
    PAIRED_WEIGHT: float=.7
    SEMANTIC_WEIGHT: float=.3
    PATIENCE: int=5
    SEED: int=42
    MAX_TRAIN_SAMPLES: int|None=None
    MAX_EVAL_SAMPLES: int|None=None
    SIM_CHUNK: int=512
    GALLERY_EXPORT_SIZE: int=256
    RESUME: bool=True
    RUN_TRAINING: bool=True
    QUICK_MODE: bool=False

cfg=Config()
cfg.S1_ROOT=cfg.DATA_ROOT/"BigEarthNet-S1"; cfg.S2_ROOT=cfg.DATA_ROOT/"BigEarthNet-S2"
cfg.METADATA=cfg.DATA_ROOT/"ben_subset.csv"; cfg.CKPT_DIR=cfg.OUTPUT_DIR/"checkpoints"
cfg.RESULTS_DIR=cfg.OUTPUT_DIR/"results"
if cfg.QUICK_MODE:
    cfg.EPOCHS=2; cfg.MAX_TRAIN_SAMPLES=1024; cfg.MAX_EVAL_SAMPLES=512
for p in (cfg.OUTPUT_DIR,cfg.CKPT_DIR,cfg.RESULTS_DIR): p.mkdir(parents=True,exist_ok=True)
RUN_NAME=f"{cfg.MODEL_NAME}_d{cfg.EMBED_DIM}_seed{cfg.SEED}"; RUN_DIR=cfg.RESULTS_DIR/RUN_NAME; RUN_DIR.mkdir(exist_ok=True)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED); torch.cuda.manual_seed_all(cfg.SEED)
print("Device:",DEVICE,"Run:",RUN_NAME)

## 2. ConfigILM preprocessing and metadata validation

In [ ]:
# The v0.1.1 model cards explicitly require this legacy order. New v0.2.0 models use the newer order.
if cfg.USE_HF_REBEN and cfg.HF_S1_REPO.endswith("v0.1.1"):
    S1_BANDS=["VH","VV"]
    S2_BANDS=["B02","B03","B04","B08","B05","B06","B07","B11","B12","B8A"]
else:
    S1_BANDS=["VV","VH"]
    S2_BANDS=["B02","B03","B04","B05","B06","B07","B08","B8A","B11","B12"]
LABELS=["Urban fabric","Industrial or commercial units","Arable land","Permanent crops","Pastures","Complex cultivation patterns","Land principally occupied by agriculture, with significant areas of natural vegetation","Agro-forestry areas","Broad-leaved forest","Coniferous forest","Mixed forest","Natural grassland and sparsely vegetated areas","Moors, heathland and sclerophyllous vegetation","Transitional woodland, shrub","Beaches, dunes, sands","Inland wetlands","Coastal wetlands","Inland waters","Marine waters"]
MEAN={"VV":-12.6438637,"VH":-19.3525581,"B01":361.0768,"B02":438.3721,"B03":614.0557,"B04":588.4096,"B05":942.8433,"B06":1769.9316,"B07":2049.5515,"B08":2193.2920,"B8A":2235.5566,"B09":2241.4553,"B11":1568.2268,"B12":997.7325}
STD={"VV":5.1334939,"VH":5.5905056,"B01":575.0687,"B02":607.0269,"B03":603.2968,"B04":684.5688,"B05":738.4327,"B06":1100.4561,"B07":1275.8054,"B08":1369.3717,"B8A":1356.5441,"B09":1316.3933,"B11":1070.1613,"B12":813.5276}
L2I={x:i for i,x in enumerate(LABELS)}
ALIASES={"patch":["patch_id","id","sample_id","s2_id","s2_name"],"s1":["s1_id","s1_name","sentinel1","sar_id"],"s2":["s2_id","s2_name","patch_id","sentinel2","optical_id"],"labels":["labels","label","new_labels","class_labels","target"],"split":["split","subset","partition","set"]}

def col(df,n,required=False):
    d={str(x).lower():x for x in df.columns}
    out=next((d[x] for x in ALIASES[n] if x in d),None)
    if required and out is None: raise ValueError(f"Missing {n} column. Found {list(df.columns)}; accepted {ALIASES[n]}")
    return out
def parse_labels(v):
    if isinstance(v,(list,tuple)): a=list(v)
    else:
        try: a=json.loads(str(v))
        except:
            try: a=ast.literal_eval(str(v))
            except: a=[x.strip() for x in str(v).replace("|",";").split(";")]
    if not isinstance(a,(list,tuple)): a=[a]
    out=[]
    for x in a:
        if str(x) in L2I: out.append(str(x))
        elif str(x).isdigit() and int(x)<19: out.append(LABELS[int(x)])
    return sorted(set(out),key=LABELS.index)
def s1_path(s1_name):
    """Build the S1 patch path without recursively scanning the dataset."""
    parent="_".join(str(s1_name).split("_")[:-3])
    return cfg.S1_ROOT/parent/str(s1_name)
def s2_path(patch_id):
    """Build the S2 patch path without recursively scanning the dataset."""
    parent="_".join(str(patch_id).split("_")[:-2])
    return cfg.S2_ROOT/parent/str(patch_id)
def band_map(folder,bands):
    out={}
    for p in list(folder.glob("*.tif"))+list(folder.glob("*.tiff")):
        s=p.stem.upper()
        for b in sorted(bands,key=len,reverse=True):
            if s==b or s.endswith("_"+b) or "_"+b+"_" in s: out[b]=p; break
    return out
def json_labels(folder):
    for p in folder.glob("*.json"):
        try:
            d=json.loads(p.read_text())
            for k in ("new_labels","labels","class_labels"):
                if k in d and parse_labels(d[k]): return parse_labels(d[k])
        except: pass
    return []
def split_for(x):
    u=int(hashlib.md5(f"{cfg.SEED}:{x}".encode()).hexdigest()[:8],16)/0xffffffff
    return "train" if u<.8 else ("val" if u<.9 else "test")

raw=pd.read_csv(cfg.METADATA); display(raw.head()); print(raw.shape,list(raw.columns))
C={k:col(raw,k,k=="patch") for k in ALIASES}; print("Detected:",C)
if not cfg.S1_ROOT.exists(): raise FileNotFoundError(cfg.S1_ROOT)
if not cfg.S2_ROOT.exists(): raise FileNotFoundError(cfg.S2_ROOT)
rows=[]; rejected=defaultdict(int); missing_s1=[]; missing_s2=[]
for _,r in tqdm(raw.iterrows(),total=len(raw),desc="Validating"):
    pid=str(r[C["patch"]]).strip(); a=str(r[C["s1"]]).strip() if C["s1"] else pid; b=str(r[C["s2"]]).strip() if C["s2"] else pid
    p1,p2=s1_path(a),s2_path(b)
    if not p1.exists(): rejected["missing_s1"]+=1; missing_s1.append({"s1_name":a,"s1_path":str(p1)}); continue
    if not p2.exists(): rejected["missing_s2"]+=1; missing_s2.append({"patch_id":b,"s2_path":str(p2)}); continue
    f1,f2=band_map(p1,S1_BANDS),band_map(p2,S2_BANDS)
    if set(f1)!=set(S1_BANDS): rejected["s1_bands"]+=1; continue
    if set(f2)!=set(S2_BANDS): rejected["s2_bands"]+=1; continue
    labs=parse_labels(r[C["labels"]]) if C["labels"] else json_labels(p2)
    if not labs: labs=json_labels(p1)
    if not labs: rejected["labels"]+=1; continue
    sp=str(r[C["split"]]).lower() if C["split"] else split_for(pid)
    sp={"validation":"val","valid":"val","training":"train","testing":"test"}.get(sp,sp)
    if sp not in ("train","val","test"): sp=split_for(pid)
    rows.append(dict(patch_id=pid,s1_id=a,s2_id=b,s1_files=f1,s2_files=f2,labels=labs,split=sp))
meta=pd.DataFrame(rows)
pd.DataFrame(missing_s1).to_csv(RUN_DIR/"missing_s1.csv",index=False)
pd.DataFrame(missing_s2).to_csv(RUN_DIR/"missing_s2.csv",index=False)
if meta.empty: raise RuntimeError(f"No valid pairs. Rejected: {dict(rejected)}")
print("Valid:",len(meta),"Rejected:",dict(rejected)); display(meta.split.value_counts())

## 3. Data loading, augmentation, and visualization

In [ ]:
def read_stack(paths,bands):
    arr=[]
    for b in bands:
        with rasterio.open(paths[b]) as src:
            arr.append(src.read(1,out_shape=(cfg.IMAGE_SIZE,cfg.IMAGE_SIZE),out_dtype="float32",resampling=Resampling.nearest))
    x=torch.from_numpy(np.stack(arr)); mu=torch.tensor([MEAN[b] for b in bands])[:,None,None]; sd=torch.tensor([STD[b] for b in bands])[:,None,None]
    return (x-mu)/sd
def yvec(labs):
    y=torch.zeros(19)
    for x in labs: y[L2I[x]]=1
    return y
def aug(a,b):
    if random.random()<.5: a,b=a.flip(-1),b.flip(-1)
    if random.random()<.5: a,b=a.flip(-2),b.flip(-2)
    k=random.randrange(4); return torch.rot90(a,k,(-2,-1)),torch.rot90(b,k,(-2,-1))
class BEN(Dataset):
    def __init__(self,frame,train=False): self.f=frame.reset_index(drop=True); self.train=train
    def __len__(self): return len(self.f)
    def __getitem__(self,i):
        r=self.f.iloc[i]; a=read_stack(r.s1_files,S1_BANDS); b=read_stack(r.s2_files,S2_BANDS)
        if self.train: a,b=aug(a,b)
        return {"s1":a,"s2":b,"labels":yvec(r.labels),"patch_id":r.patch_id}
def limit(x,n): return x if n is None or len(x)<=n else x.sample(n,random_state=cfg.SEED)
tr=limit(meta[meta.split=="train"],cfg.MAX_TRAIN_SAMPLES); va=limit(meta[meta.split=="val"],cfg.MAX_EVAL_SAMPLES); te=limit(meta[meta.split=="test"],cfg.MAX_EVAL_SAMPLES)
if min(len(va),len(te))==0:
    z=meta.sample(frac=1,random_state=cfg.SEED); n=len(z); tr,va,te=z[:int(.8*n)],z[int(.8*n):int(.9*n)],z[int(.9*n):]
train_ds,val_ds,test_ds=BEN(tr,True),BEN(va),BEN(te)
kw=dict(num_workers=cfg.NUM_WORKERS,pin_memory=DEVICE.type=="cuda",persistent_workers=cfg.NUM_WORKERS>0)
train_loader=DataLoader(train_ds,cfg.BATCH_SIZE,shuffle=True,drop_last=len(train_ds)>=cfg.BATCH_SIZE,**kw)
val_loader=DataLoader(val_ds,cfg.EVAL_BATCH_SIZE,shuffle=False,**kw); test_loader=DataLoader(test_ds,cfg.EVAL_BATCH_SIZE,shuffle=False,**kw)
s=train_ds[0]; assert s["s1"].shape==(2,120,120) and s["s2"].shape==(12,120,120) and torch.isfinite(s["s2"]).all()
def raw(x,bands): return (x*torch.tensor([STD[b] for b in bands])[:,None,None]+torch.tensor([MEAN[b] for b in bands])[:,None,None]).numpy()
def rgb(x):
    z=raw(x,S2_BANDS); a=z[[S2_BANDS.index("B04"),S2_BANDS.index("B03"),S2_BANDS.index("B02")]]; lo,hi=np.percentile(a,[2,98]); return np.clip((np.moveaxis(a,0,-1)-lo)/(hi-lo+1e-6),0,1)
def sar(x):
    a=raw(x,S1_BANDS)[0]; lo,hi=np.percentile(a,[2,98]); return np.clip((a-lo)/(hi-lo+1e-6),0,1)
fig,ax=plt.subplots(3,2,figsize=(9,12))
for i in range(3):
    q=train_ds[i]; ax[i,0].imshow(sar(q["s1"]),cmap="gray"); ax[i,1].imshow(rgb(q["s2"]))
    ax[i,0].set_title("S1 "+q["patch_id"]); ax[i,1].set_title("S2 paired RGB")
    ax[i,0].axis("off"); ax[i,1].axis("off")
plt.tight_layout(); plt.savefig(RUN_DIR/"data_check.png",dpi=140); plt.show()
print("train/val/test",len(tr),len(va),len(te))

## 4. Model, weight fallback, and hybrid contrastive training

In [ ]:
REGISTRY={"resnet18":"resnet18","resnet34":"resnet34","resnet50":"resnet50","efficientnet_b0":"efficientnet_b0","convnext_tiny":"convnext_tiny","vit_small_patch16_224":"vit_small_patch16_224","swin_tiny_patch4_window7_224":"swin_tiny_patch4_window7_224"}
def backbone(ch,pretrained):
    name=REGISTRY.get(cfg.MODEL_NAME,cfg.MODEL_NAME); kw=dict(pretrained=pretrained,in_chans=ch,num_classes=0)
    if any(x in name for x in ("vit","swin","beit","deit")): kw["img_size"]=cfg.IMAGE_SIZE
    try: return timm.create_model(name,**kw),("timm_pretrained" if pretrained else "random")
    except Exception as e:
        print("Pretrained failed; training from random:",e); kw["pretrained"]=False; return timm.create_model(name,**kw),"random_fallback"
class Encoder(nn.Module):
    def __init__(self,b,d):
        super().__init__(); self.backbone=b; self.projector=nn.Sequential(nn.Linear(b.num_features,d),nn.GELU(),nn.LayerNorm(d),nn.Linear(d,d))
    def forward(self,x):
        z=self.backbone(x)
        if z.ndim>2: z=z.mean(tuple(range(2,z.ndim)))
        return F.normalize(self.projector(z),dim=-1)
class Dual(nn.Module):
    def __init__(self):
        super().__init__(); generic=cfg.USE_GENERIC_PRETRAINED and not cfg.USE_HF_REBEN; b1,a=backbone(len(S1_BANDS),generic); b2,b=backbone(len(S2_BANDS),generic)
        self.s1_encoder=Encoder(b1,cfg.EMBED_DIM); self.s2_encoder=Encoder(b2,cfg.EMBED_DIM); self.initialization={"s1":a,"s2":b}
    def forward(self,a,b): return self.s1_encoder(a),self.s2_encoder(b)
def local_load(model,path,min_coverage=.2):
    report={"path":path,"accepted":False,"coverage":0.}
    if not path or not Path(path).exists(): report["reason"]="absent; fallback retained"; return report
    d=torch.load(path,map_location="cpu")
    for k in ("state_dict","model_state_dict","model"):
        if isinstance(d,dict) and k in d: d=d[k]
    target=model.state_dict(); matched={}
    clean=lambda x:x.replace("module.","").replace("model.","")
    src={clean(k):v for k,v in d.items() if torch.is_tensor(v)}
    for k,v in target.items():
        cand=[x for x,w in src.items() if w.shape==v.shape and (clean(k).endswith(x) or x.endswith(clean(k)))]
        if cand: matched[k]=src[cand[0]]
    report["coverage"]=sum(v.numel() for v in matched.values())/sum(v.numel() for v in target.values())
    if report["coverage"]>=min_coverage:
        model.load_state_dict(matched,strict=False); model.initialization={"s1":"local_BEN_MM","s2":"local_BEN_MM"}; report["accepted"]=True
    report["reason"]="loaded" if report["accepted"] else "incompatible; fallback retained"
    return report
def reben_file(repo,filename,modality):
    if cfg.HF_LOCAL_DIR:
        path=Path(cfg.HF_LOCAL_DIR)/modality/filename
        if not path.exists(): raise FileNotFoundError(f"Missing local reBEN file: {path}")
        return str(path)
    return hf_hub_download(repo_id=repo,filename=filename,token=cfg.HF_TOKEN)

def load_reben_encoder(encoder,repo,modality):
    config_path=reben_file(repo,"config.json",modality)
    weights_path=reben_file(repo,"model.safetensors",modality)
    info=json.loads(Path(config_path).read_text())
    expected=len(S1_BANDS) if modality=="s1" else len(S2_BANDS)
    if info["timm_model_name"]!=cfg.MODEL_NAME:
        raise ValueError(f"{repo} requires {info['timm_model_name']}, not {cfg.MODEL_NAME}")
    if info["channels"]!=expected:
        raise ValueError(f"{repo} requires {info['channels']} channels, pipeline has {expected}")
    state=load_safetensors(weights_path,device="cpu")
    prefixes=("model.vision_encoder.","vision_encoder.")
    vision={}
    for key,value in state.items():
        for prefix in prefixes:
            if key.startswith(prefix):
                vision[key[len(prefix):]]=value
                break
    target=encoder.backbone.state_dict()
    matched={k:v for k,v in vision.items() if k in target and v.shape==target[k].shape}
    coverage=sum(v.numel() for v in matched.values())/max(sum(v.numel() for v in target.values()),1)
    encoder.backbone.load_state_dict(matched,strict=False)
    if coverage<.95: raise RuntimeError(f"Unsafe {modality} checkpoint coverage: {coverage:.1%}")
    return {"repo":repo,"coverage":coverage,"matched":len(matched),"config":info}

model=Dual()
if cfg.USE_HF_REBEN:
    s1_report=load_reben_encoder(model.s1_encoder,cfg.HF_S1_REPO,"s1")
    s2_report=load_reben_encoder(model.s2_encoder,cfg.HF_S2_REPO,"s2")
    model.initialization={"s1":"HF_reBEN_v0.1.1","s2":"HF_reBEN_v0.1.1"}
    load_report={"accepted":True,"coverage":min(s1_report["coverage"],s2_report["coverage"]),"s1":s1_report,"s2":s2_report}
else:
    load_report=local_load(model,cfg.LOCAL_PRETRAINED_PATH)
model.to(DEVICE)
print(model.initialization,json.dumps(load_report,indent=2,default=str))

def loss_fn(a,b,y):
    logits=a@b.T/cfg.TEMPERATURE; t=torch.arange(len(a),device=a.device)
    paired=(F.cross_entropy(logits,t)+F.cross_entropy(logits.T,t))/2
    z=torch.cat([a,b]); yy=torch.cat([y,y]); sim=z@z.T/cfg.TEMPERATURE
    eye=torch.eye(len(z),device=z.device,dtype=torch.bool); pos=(yy@yy.T>0)&~eye
    sim=sim-sim.max(1,keepdim=True).values.detach(); ex=torch.exp(sim)*~eye
    valid=pos.any(1); semantic=-torch.log(((ex*pos).sum(1).clamp_min(1e-12)/ex.sum(1).clamp_min(1e-12))[valid]).mean()
    return cfg.PAIRED_WEIGHT*paired+cfg.SEMANTIC_WEIGHT*semantic,paired,semantic
@torch.no_grad()
def val_loss():
    model.eval(); total=n=0
    for q in val_loader:
        a,b,y=q["s1"].to(DEVICE),q["s2"].to(DEVICE),q["labels"].to(DEVICE); z1,z2=model(a,b); l,_,_=loss_fn(z1,z2,y)
        total+=l.item()*len(a); n+=len(a)
    return total/max(n,1)
BEST=cfg.CKPT_DIR/f"{RUN_NAME}_best.pth"; LAST=cfg.CKPT_DIR/f"{RUN_NAME}_last.pth"; history=[]
def train():
    global history
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.LR,weight_decay=cfg.WEIGHT_DECAY); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,cfg.EPOCHS)
    scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda"); start=0; best=float("inf"); stale=0
    if cfg.RESUME and LAST.exists():
        p=torch.load(LAST,map_location=DEVICE); model.load_state_dict(p["model_state_dict"]); opt.load_state_dict(p["optimizer"]); sch.load_state_dict(p["scheduler"]); history=p["history"]; start=p["epoch"]+1; best=p["best"]
    for ep in range(start,cfg.EPOCHS):
        model.train(); total=n=0
        for q in tqdm(train_loader,desc=f"epoch {ep+1}"):
            a,b,y=q["s1"].to(DEVICE),q["s2"].to(DEVICE),q["labels"].to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type,dtype=torch.float16,enabled=DEVICE.type=="cuda"): z1,z2=model(a,b); l,pl,sl=loss_fn(z1,z2,y)
            scaler.scale(l).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.); scaler.step(opt); scaler.update()
            total+=l.item()*len(a); n+=len(a)
        v=val_loss(); sch.step(); row={"epoch":ep+1,"train_loss":total/n,"val_loss":v,"lr":opt.param_groups[0]["lr"]}; history.append(row)
        payload={"epoch":ep,"model_state_dict":model.state_dict(),"optimizer":opt.state_dict(),"scheduler":sch.state_dict(),"history":history,"best":min(best,v),"config":{k:str(x) if isinstance(x,Path) else x for k,x in asdict(cfg).items()},"initialization":model.initialization}
        torch.save(payload,LAST)
        if v<best: best=v; stale=0; torch.save(payload,BEST)
        else: stale+=1
        pd.DataFrame(history).to_csv(RUN_DIR/"history.csv",index=False); print(row)
        if stale>=cfg.PATIENCE: break
if cfg.RUN_TRAINING: train()
if BEST.exists(): model.load_state_dict(torch.load(BEST,map_location=DEVICE)["model_state_dict"])
if history:
    pd.DataFrame(history).plot(x="epoch",y=["train_loss","val_loss"],marker="o"); plt.grid(alpha=.3); plt.savefig(RUN_DIR/"history.png",dpi=150); plt.show()

## 5. Retrieval evaluation and experiment comparison

In [ ]:
@torch.no_grad()
def embeddings():
    model.eval(); a=[]; b=[]; y=[]; ids=[]; tic=time.perf_counter()
    for q in tqdm(test_loader):
        z1,z2=model(q["s1"].to(DEVICE),q["s2"].to(DEVICE)); a.append(z1.cpu()); b.append(z2.cpu()); y.append(q["labels"]); ids+=q["patch_id"]
    return {"s1":torch.cat(a),"s2":torch.cat(b),"labels":torch.cat(y),"ids":ids,"throughput":2*len(ids)/(time.perf_counter()-tic)}
E=embeddings()
D={"s1_to_s1":("s1","s1",True),"s2_to_s2":("s2","s2",True),"s1_to_s2":("s1","s2",False),"s2_to_s1":("s2","s1",False)}
directions=list(D) if cfg.TASK_MODE=="all" else [cfg.TASK_MODE]
def evaluate(name):
    qn,gn,same=D[name]; q,g,y=E[qn],E[gn],E["labels"]; sums=defaultdict(float); preds=[]; lat=[]
    maxk=min(10,len(g)-(1 if same else 0)); gd=g.to(DEVICE); yd=y.to(DEVICE)
    for st in range(0,len(q),cfg.SIM_CHUNK):
        en=min(st+cfg.SIM_CHUNK,len(q)); qq=q[st:en].to(DEVICE); qy=y[st:en].to(DEVICE)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        tic=time.perf_counter(); score=qq@gd.T
        if same: score[torch.arange(en-st),torch.arange(st,en,device=DEVICE)]=-torch.inf
        val,ind=score.topk(maxk,dim=1)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        lat += [(time.perf_counter()-tic)/(en-st)]*(en-st)
        allrel=qy@yd.T>0
        if same: allrel[torch.arange(en-st),torch.arange(st,en,device=DEVICE)]=False
        rel=allrel.gather(1,ind); total=allrel.sum(1).clamp_min(1)
        for requested in (5,10):
            k=min(requested,maxk); hit=rel[:,:k].sum(1).float(); p=hit/k; r=hit/total; f=2*p*r/(p+r).clamp_min(1e-12)
            rr=rel[:,:k].float(); pp=rr.cumsum(1)/torch.arange(1,k+1,device=DEVICE); ap=(pp*rr).sum(1)/torch.minimum(total,torch.tensor(k,device=DEVICE))
            for key,x in ((f"precision@{requested}",p),(f"recall@{requested}",r),(f"f1@{requested}",f),(f"map@{requested}",ap)): sums[key]+=x.sum().item()
        for i in range(en-st): preds.append({"direction":name,"query_id":E["ids"][st+i],"gallery_ids":[E["ids"][j] for j in ind[i].cpu().tolist()],"scores":val[i].cpu().tolist(),"relevant":rel[i].cpu().tolist()})
    out={k:v/len(q) for k,v in sums.items()}; out.update(direction=name,mean_latency_ms=np.mean(lat)*1000,median_latency_ms=np.median(lat)*1000,embedding_throughput=E["throughput"]); return out,preds
metrics=[]; predictions=[]
for d in directions:
    a,b=evaluate(d); metrics.append(a); predictions+=b
M=pd.DataFrame(metrics); display(M); M.to_csv(RUN_DIR/"retrieval_metrics.csv",index=False)
with open(RUN_DIR/"top10_predictions.jsonl","w") as f:
    for x in predictions: f.write(json.dumps(x)+"\n")
summary={"run_name":RUN_NAME,"model":cfg.MODEL_NAME,"initialization":json.dumps(model.initialization),"weight_coverage":load_report["coverage"],"f1@5":M["f1@5"].mean(),"f1@10":M["f1@10"].mean(),"map@10":M["map@10"].mean(),"latency_ms":M["mean_latency_ms"].mean(),"epochs":len(history)}
P=cfg.RESULTS_DIR/"experiment_summary.csv"; old=pd.read_csv(P) if P.exists() else pd.DataFrame()
old=old[old.run_name!=RUN_NAME] if len(old) and "run_name" in old else old; comparison=pd.concat([old,pd.DataFrame([summary])],ignore_index=True).sort_values("f1@10",ascending=False); comparison.to_csv(P,index=False); display(comparison)
# Acceptance checks: deterministic inference, self exclusion, checkpoint strict reload.
model.eval()
with torch.no_grad(): x1=model.s1_encoder(s["s1"].unsqueeze(0).to(DEVICE)); x2=model.s1_encoder(s["s1"].unsqueeze(0).to(DEVICE))
assert torch.allclose(x1,x2,atol=1e-6)
for p in predictions:
    if p["direction"] in ("s1_to_s1","s2_to_s2"): assert p["query_id"] not in p["gallery_ids"]
print("Smoke tests passed.")

## 6. Hugging Face Gradio Space ZIP

In [ ]:
SPACE=cfg.OUTPUT_DIR/"bigearthnet_retrieval_space"; shutil.rmtree(SPACE,ignore_errors=True); (SPACE/"previews").mkdir(parents=True); (SPACE/"examples").mkdir()
idx=np.linspace(0,len(test_ds)-1,min(cfg.GALLERY_EXPORT_SIZE,len(test_ds)),dtype=int); rows=[]; ga=[]; gb=[]
for j,i in enumerate(idx):
    x=test_ds[int(i)]; ga.append(E["s1"][i].numpy()); gb.append(E["s2"][i].numpy())
    n1=f"previews/s1_{j:04d}.png"; n2=f"previews/s2_{j:04d}.png"; Image.fromarray((sar(x["s1"])*255).astype("uint8")).save(SPACE/n1); Image.fromarray((rgb(x["s2"])*255).astype("uint8")).save(SPACE/n2)
    rows.append({"patch_id":x["patch_id"],"labels":json.dumps([LABELS[k] for k in x["labels"].nonzero().flatten()]),"s1_preview":n1,"s2_preview":n2})
np.savez_compressed(SPACE/"gallery_embeddings.npz",s1=np.stack(ga),s2=np.stack(gb)); pd.DataFrame(rows).to_csv(SPACE/"gallery_metadata.csv",index=False)
ex=test_ds[0]; np.savez_compressed(SPACE/"examples/s1_example.npz",image=raw(ex["s1"],S1_BANDS)); np.savez_compressed(SPACE/"examples/s2_example.npz",image=raw(ex["s2"],S2_BANDS))
ck=BEST if BEST.exists() else LAST
if not ck.exists(): torch.save({"model_state_dict":model.state_dict()},ck)
shutil.copy2(ck,SPACE/"best_model.pth")
conf={"model_name":cfg.MODEL_NAME,"embed_dim":cfg.EMBED_DIM,"image_size":cfg.IMAGE_SIZE,"s1_bands":S1_BANDS,"s2_bands":S2_BANDS,"mean":MEAN,"std":STD}
(SPACE/"config.json").write_text(json.dumps(conf,indent=2)); (SPACE/"labels.json").write_text(json.dumps(LABELS,indent=2))
MODEL_CODE=r'''import json
from pathlib import Path
import numpy as np, rasterio, timm, torch
from rasterio.enums import Resampling
import torch.nn as nn
import torch.nn.functional as F
R=Path(__file__).parent; C=json.loads((R/"config.json").read_text()); D=torch.device("cuda" if torch.cuda.is_available() else "cpu")
def bb(ch):
 k=dict(pretrained=False,in_chans=ch,num_classes=0)
 if any(x in C["model_name"] for x in ("vit","swin","beit","deit")): k["img_size"]=C["image_size"]
 return timm.create_model(C["model_name"],**k)
class E(nn.Module):
 def __init__(self,b): super().__init__(); self.backbone=b; self.projector=nn.Sequential(nn.Linear(b.num_features,C["embed_dim"]),nn.GELU(),nn.LayerNorm(C["embed_dim"]),nn.Linear(C["embed_dim"],C["embed_dim"]))
 def forward(self,x):
  z=self.backbone(x)
  if z.ndim>2: z=z.mean(tuple(range(2,z.ndim)))
  return F.normalize(self.projector(z),dim=-1)
class M(nn.Module):
 def __init__(self): super().__init__(); self.s1_encoder=E(bb(len(C["s1_bands"]))); self.s2_encoder=E(bb(len(C["s2_bands"])))
M0=M(); p=torch.load(R/"best_model.pth",map_location="cpu"); M0.load_state_dict(p.get("model_state_dict",p)); M0.to(D).eval()
def load(path,mod):
 bands=C["s1_bands"] if mod=="S1" else C["s2_bands"]; path=Path(path)
 if path.suffix.lower()==".npz":
  z=np.load(path); a=z["image"] if "image" in z else z[z.files[0]]
 else:
  with rasterio.open(path) as s: a=s.read(out_shape=(s.count,C["image_size"],C["image_size"]),out_dtype="float32",resampling=Resampling.nearest)
 a=np.asarray(a,dtype="float32")
 if a.ndim==3 and a.shape[-1]==len(bands): a=np.moveaxis(a,-1,0)
 if a.shape[0]!=len(bands): raise ValueError(f"{mod} needs {len(bands)} ordered bands")
 if a.shape[-2:]!=(C["image_size"],C["image_size"]): a=F.interpolate(torch.from_numpy(a)[None],size=(C["image_size"],C["image_size"]),mode="nearest")[0].numpy()
 mu=np.array([C["mean"][b] for b in bands])[:,None,None]; sd=np.array([C["std"][b] for b in bands])[:,None,None]
 return torch.tensor((a-mu)/sd,dtype=torch.float32)[None]
@torch.inference_mode()
def embed(path,mod): return (M0.s1_encoder if mod=="S1" else M0.s2_encoder)(load(path,mod).to(D)).cpu().numpy()[0]
'''
APP_CODE=r'''from pathlib import Path
import shutil, tempfile, zipfile
import gradio as gr, numpy as np, pandas as pd
from model import embed
R=Path(__file__).parent; meta=pd.read_csv(R/"gallery_metadata.csv"); em=np.load(R/"gallery_embeddings.npz")
def user_gallery(path,mod):
 temp=Path(tempfile.mkdtemp(prefix="ben_gallery_"))
 with zipfile.ZipFile(path) as z:
  for item in z.infolist():
   target=(temp/item.filename).resolve()
   if temp.resolve() not in target.parents and target!=temp.resolve(): raise ValueError("Unsafe ZIP path")
  z.extractall(temp)
 files=sorted(x for x in temp.rglob("*") if x.suffix.lower() in (".npz",".tif",".tiff"))
 vectors=[]; names=[]
 for x in files:
  try: vectors.append(embed(x,mod)); names.append(x.stem)
  except Exception as exc: print("Skipping",x,exc)
 if not vectors: shutil.rmtree(temp,ignore_errors=True); raise ValueError("No valid full-band gallery files")
 return np.stack(vectors),names,temp
def search(q,qmod,gmod,k,gzip):
 if q is None: raise gr.Error("Upload full-band NPZ or GeoTIFF")
 z=embed(q,qmod); cleanup=None
 if gzip:
  vectors,names,cleanup=user_gallery(gzip,gmod); scores=vectors@z; ix=np.argsort(-scores)[:int(k)]
  table=[[n+1,names[i],float(scores[i]),"user gallery"] for n,i in enumerate(ix)]; gallery=[]
 else:
  scores=em[gmod.lower()]@z; ix=np.argsort(-scores)[:int(k)]; pc="s1_preview" if gmod=="S1" else "s2_preview"
  table=[[n+1,meta.iloc[i].patch_id,float(scores[i]),meta.iloc[i].labels] for n,i in enumerate(ix)]
  gallery=[(str(R/meta.iloc[i][pc]),f"#{n+1} {meta.iloc[i].patch_id}") for n,i in enumerate(ix)]
 if cleanup: shutil.rmtree(cleanup,ignore_errors=True)
 return table,gallery
with gr.Blocks() as demo:
 gr.Markdown("# BigEarthNet Cross-Modal Retrieval")
 with gr.Row(): q=gr.File(label="Full-band NPZ/GeoTIFF"); gzip=gr.File(label="Optional gallery ZIP")
 with gr.Row(): qm=gr.Radio(["S1","S2"],value="S2",label="Query"); gm=gr.Radio(["S1","S2"],value="S1",label="Gallery"); k=gr.Slider(1,10,5,step=1)
 go=gr.Button("Retrieve"); out=gr.Dataframe(headers=["Rank","Patch ID","Score","Labels"]); gal=gr.Gallery(columns=5)
 gr.Examples([[str(R/"examples/s2_example.npz"),"S2","S1",5,None],[str(R/"examples/s1_example.npz"),"S1","S2",5,None]],inputs=[q,qm,gm,k,gzip])
 go.click(search,[q,qm,gm,k,gzip],[out,gal])
demo.launch()
'''
(SPACE/"model.py").write_text(MODEL_CODE); (SPACE/"app.py").write_text(APP_CODE)
(SPACE/"requirements.txt").write_text("torch>=2.2\ntimm>=1.0.15\ngradio>=4.44\nrasterio>=1.3\npandas>=2\nnumpy>=1.24\n")
(SPACE/"README.md").write_text("---\ntitle: BigEarthNet Cross-Modal Retrieval\nemoji: 🛰️\nsdk: gradio\napp_file: app.py\npinned: false\n---\n\nS1/S2 retrieval demo using a fine-tuned reBEN ViT checkpoint, with a bundled compact gallery and full-band upload examples.")
ZIP=cfg.OUTPUT_DIR/"bigearthnet_retrieval_space.zip"
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in SPACE.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(SPACE))
with zipfile.ZipFile(ZIP) as z:
    required={"app.py","model.py","best_model.pth","config.json","gallery_embeddings.npz","gallery_metadata.csv","README.md","requirements.txt"}
    assert not required-set(z.namelist()) and z.testzip() is None
print("Ready:",ZIP,ZIP.stat().st_size/1e6,"MB")